# ADR-1: Collection Data Structure for Key Value store

Decision between "2 dictionary" or "3 dictionary" model for key-value store to support indexing and searching while keeping memory footprint low and performance high.

let's consider the following example is what I want to store and index is optional:
```
key = "key-1"
data = "data-1"
index = {"index-1": "index-1-value", "index-2": "index-2-value"}
```


# 2 dict method

1- Method: using a dictionary for key-value and a second dictionary for querying index values.
index will be stored in a tuple with value. this will help creating less overhead
```
data = {key: (data, {index_name: index_value})}
index = {index_name:{index_value: {key}}}
```

In [2]:
class NeoCollectionV1:
    def __init__(self, collection_name):
        self.name = collection_name
        self.data = dict()
        self.index_data = dict()
        # self._lock = threading.Lock()
        # self._last_id = 0

    def put(self, key, value, indexes=None):
        self._delete_indexes(key)
        self.data[key] = (value, indexes)
        if indexes:
            for index_name, index_value in indexes.items():
                self.index_data.setdefault(index_name, dict()).setdefault(index_value, set()).add(key)
        return True

    def get(self, key):
        record_value, _ = self.data.get(key, (None, None))
        return record_value

    def _delete_indexes(self, key):
        _, indexes = self.data.get(key, (None, None))
        if indexes:
            for index_name, index_value in indexes.items():
                # not using discard or safe dict lookups if the data is corrupt
                # TODO: may be raise an error with data corruption message?
                self.index_data[index_name][index_value].remove(key)
                if len(self.index_data[index_name][index_value]) == 0:
                    del self.index_data[index_name][index_value]
                if len(self.index_data[index_name]) == 0:
                    del self.index_data[index_name]

    def delete(self, key):
        self._delete_indexes(key)
        if self.data.pop(key, None):
            return True
        return False

# 3 dict method

1 - Method: using a dictionary for key - value and a second dictionary for querying index values and 3rd dictionary for storing reverse index lookup for fast deletion.
data and index will be stored in separate dicts. a 3rd dict will help for fast deletes.

```
records = {key: data}
index = {index_name:{index_value: {key}}}
cleanup_index =  {key:{index_name: index_value}}
```

In [4]:
class NeoCollectionV2:
    def __init__(self, collection_name):
        self.name = collection_name
        self.records = dict()
        self.indexes = dict()
        self.reverse_indexes = dict()

    def get(self, key):
        record_value = self.records.get(key, None)
        return record_value

    def put(self, key, value, indexes=None):
        self._delete_indexes(key)
        self.records[key] = value
        if indexes:
            for index_name, index_value in indexes.items():
                self.indexes.setdefault(index_name, dict()).setdefault(index_value, set()).add(key)
                self.reverse_indexes.setdefault(key, dict())[index_name] = index_value

    def _delete_indexes(self, key):
        index_to_remove = self.reverse_indexes.pop(key, None)
        if index_to_remove:
            for index_name, index_value in index_to_remove.items():
                if index_name in self.indexes and index_value in self.indexes[index_name]:
                    self.indexes[index_name][index_value].discard(key)
                    if not self.indexes[index_name][index_value]:
                        del self.indexes[index_name][index_value]
                    if not self.indexes[index_name]:
                        del self.indexes[index_name]

    def delete(self, key):
        self._delete_indexes(key)
        return self.records.pop(key, None) is not None

Now lets compare memory usage and cpu time consumed by each approach

In [11]:
from eprofiler import profile


def test_collection(collection, collection_name, insert_count=1000):
    coll = collection(collection_name)
    for x in range(insert_count):
        key = f"key-{x}"
        data = f"data-{x}"
        index = { f"index-{a}": f"index-{a}_value-{a}" for a in range(10)}
        coll.put(key, data)

profiled_test_v1 = profile(test_collection, label="NeoCollectionV1")
profiled_test_v1(NeoCollectionV1, "NeoCollectionV1", insert_count=100_000)

profiled_test_v2 = profile(test_collection, label="NeoCollectionV2")
profiled_test_v2(NeoCollectionV2, "NeoCollectionV2", insert_count=100_000)

{'label': 'NeoCollectionV1', 'function': 'test_collection', 'duration': '8.718732', 'cpu_time': '8.666662', 'peak': 19415783, 'current': 2375}
{'label': 'NeoCollectionV2', 'function': 'test_collection', 'duration': '8.422356', 'cpu_time': '8.365357', 'peak': 14574238, 'current': 2044}


Finding: if I dont have indexes, NeoCollectionV2 performs slightly faster and uses less memory

In [12]:
from eprofiler import profile


def test_collection(collection, collection_name, insert_count=1000):
    coll = collection(collection_name)
    for x in range(insert_count):
        key = f"key-{x}"
        data = f"data-{x}"
        index = { f"index-{a}": f"index-{a}_value-{a}" for a in range(10)}
        coll.put(key, data, index)

profiled_test_v1 = profile(test_collection, label="NeoCollectionV1")
profiled_test_v1(NeoCollectionV1, "NeoCollectionV1", insert_count=100_000)

profiled_test_v2 = profile(test_collection, label="NeoCollectionV2")
profiled_test_v2(NeoCollectionV2, "NeoCollectionV2", insert_count=100_000)

{'label': 'NeoCollectionV1', 'function': 'test_collection', 'duration': '12.596622', 'cpu_time': '12.496500', 'peak': 192564093, 'current': 6817}
{'label': 'NeoCollectionV2', 'function': 'test_collection', 'duration': '15.712626', 'cpu_time': '15.606298', 'peak': 190921417, 'current': 6925}


Finding: if I have 10 indexes for each record, NeoCollectionV2 is slower but more memory efficient

In [13]:
from eprofiler import profile


def test_collection(collection, collection_name, insert_count=1000):
    coll = collection(collection_name)
    for x in range(insert_count):
        key = f"key-{x}"
        data = f"data-{x}"
        index = { f"index-{a}": f"index-{a}_value-{a}" for a in range(10)}
        coll.put(key, data, index)
    for x in range(0, insert_count, 2):
        coll.delete(f"key-{x}")

profiled_test_v1 = profile(test_collection, label="NeoCollectionV1")
profiled_test_v1(NeoCollectionV1, "NeoCollectionV1", insert_count=100_000)

profiled_test_v2 = profile(test_collection, label="NeoCollectionV2")
profiled_test_v2(NeoCollectionV2, "NeoCollectionV2", insert_count=100_000)

{'label': 'NeoCollectionV1', 'function': 'test_collection', 'duration': '13.910605', 'cpu_time': '13.847570', 'peak': 192656979, 'current': 109599}
{'label': 'NeoCollectionV2', 'function': 'test_collection', 'duration': '17.502765', 'cpu_time': '17.189669', 'peak': 190916752, 'current': 2228}


Finding: NeoCollectonV2 is slower on speed but better on memory with deletes on heavy indexes


In [14]:
from eprofiler import profile


def test_collection(collection, collection_name, insert_count=1000):
    coll = collection(collection_name)
    for x in range(insert_count):
        key = f"key-{x}"
        data = f"data-{x}"
        # index = { f"index-{a}": f"index-{a}_value-{a}" for a in range(10)}
        coll.put(key, data)
    for x in range(0, insert_count, 2):
        coll.delete(f"key-{x}")

profiled_test_v1 = profile(test_collection, label="NeoCollectionV1")
profiled_test_v1(NeoCollectionV1, "NeoCollectionV1", insert_count=100_000)

profiled_test_v2 = profile(test_collection, label="NeoCollectionV2")
profiled_test_v2(NeoCollectionV2, "NeoCollectionV2", insert_count=100_000)

{'label': 'NeoCollectionV1', 'function': 'test_collection', 'duration': '1.024199', 'cpu_time': '1.020203', 'peak': 19412994, 'current': 57774}
{'label': 'NeoCollectionV2', 'function': 'test_collection', 'duration': '0.865439', 'cpu_time': '0.863684', 'peak': 14571863, 'current': 949}


Finding: if there are no indexes, NeoCollectionV2 is better on both memory and speed

# Summary

NeoCollectionV2 (3 Dictionary) seems to perform better in most situations. Also having index data seperated from values will give more flexibility.